In [ ]:
!pip install spotipy tqdm pandas requests
!pip install python-dotenv
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import random, time, json
import pandas as pd
from tqdm import tqdm

from dotenv import load_dotenv
import os

load_dotenv()

SPOTIFY_CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
SPOTIFY_CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")

sp = spotipy.Spotify(
    client_credentials_manager=SpotifyClientCredentials(
        client_id=SPOTIFY_CLIENT_ID,
        client_secret=SPOTIFY_CLIENT_SECRET
    )
)


EMOTIONS = {
    "sad": ["sad", "melancholy", "heartbreak", "emotional", "crying", "depressed"],
    "happy": ["happy", "joyful", "uplifting", "feel good", "positive vibes", "good mood"],
    "chill": ["chill", "relaxing", "lo-fi", "calm", "peaceful", "ambient"],
    "energetic": ["workout", "energetic", "pump up", "high energy", "running", "gym"],
    "calm": ["calm", "meditation", "ambient", "peaceful", "sleep", "relax"],
    "party": ["party", "dance", "club", "festival", "hype", "celebration"],
    "romantic": ["romantic", "love", "wedding", "date night", "cuddle", "slow dance"],
    "focus": ["focus", "concentration", "study", "instrumental", "work", "productivity"]
}


def collect_massive_tracks(emotion, max_tracks=2000):
    print(f"\nCollecting MASSIVE tracks for emotion: {emotion.upper()}")
    tracks_data = []
    seen_ids = set()

    search_terms = EMOTIONS.get(emotion, [emotion])

    for term in search_terms:
        if len(tracks_data) >= max_tracks:
            break
        query = f"{term} playlist"
        print(f"  Searching playlists for: '{query}'")
        try:
            results = sp.search(q=query, type="playlist", limit=50)
            playlists = results.get("playlists", {}).get("items", [])
            print(f"    Found {len(playlists)} playlists")
        except Exception as e:
            print(f"    Search failed: {e}")
            continue

        for playlist in playlists:
            if playlist is None:
                continue  # Skip invalid playlist entries

            if len(tracks_data) >= max_tracks:
                break

            playlist_id = playlist.get("id")
            playlist_name = playlist.get("name", "Unknown Playlist")
            if not playlist_id:
                continue

            print(f"      Fetching tracks from: {playlist_name[:40]}...")
            offset = 0
            limit = 100

            while True:
                try:
                    playlist_items = sp.playlist_items(
                        playlist_id,
                        limit=limit,
                        offset=offset,
                        fields="items(track(id,name,artists,duration_ms))"
                    )
                    items = playlist_items.get("items") or []
                    if not items:
                        break

                    for item in items:
                        track = item.get("track")
                        if not track:
                            continue
                        track_id = track.get("id")
                        track_name = track.get("name")
                        if not track_id or track_id in seen_ids:
                            continue
                        artists = ", ".join([a.get("name", "Unknown") for a in track.get("artists", [])][:3])
                        tracks_data.append({
                            "emotion": emotion,
                            "track_id": track_id,
                            "track_name": track_name,
                            "artists": artists,
                            "playlist_name": playlist_name,
                            "duration_ms": track.get("duration_ms", random.randint(180000, 300000))
                        })
                        seen_ids.add(track_id)

                        if len(tracks_data) % 200 == 0:
                            print(f"        Collected {len(tracks_data)} tracks so far...")

                    offset += limit
                    if len(tracks_data) >= max_tracks:
                        break

                except Exception as e:
                    print(f"        Playlist fetch error: {e}")
                    break

            time.sleep(0.2)

    print(f"  Finished collecting {len(tracks_data)} tracks for {emotion}")
    return tracks_data


all_tracks = []
TARGET_PER_EMOTION = 1000 

for emotion in tqdm(EMOTIONS.keys(), desc="Collecting tracks by emotion"):
    tracks = collect_massive_tracks(emotion, TARGET_PER_EMOTION)
    all_tracks.extend(tracks)

    # Save checkpoint after each emotion
    with open("massive_music_dataset.json", "w", encoding="utf-8") as f:
        json.dump(all_tracks, f, indent=2)
    print(f"  💾 Checkpoint saved: {len(all_tracks)} tracks total")
    time.sleep(1)

print(f"\nTotal tracks collected: {len(all_tracks)}")


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



  Searching playlists for: 'sad playlist'
    Found 49 playlists
      Fetching tracks from: Sad playlist to cry to at 3am 🌃...
      Fetching tracks from: crying myself to sleep 😢 sad songs...
        Collected 200 tracks so far...
      Fetching tracks from: Sad playlist for crying...
        Collected 400 tracks so far...
      Fetching tracks from: saddest playlist on earth...
      Fetching tracks from: sad songs to cry out to 😭...
        Collected 600 tracks so far...
      Fetching tracks from: Sad songs for tik tok edits...
      Fetching tracks from: Sad playlist 😫😫...
        Collected 800 tracks so far...
      Fetching tracks from: sad playlist to cry myself to sleep 😔...
        Collected 1000 tracks so far...
  Finished collecting 1003 tracks for sad
  💾 Checkpoint saved: 1003 tracks total



  Searching playlists for: 'happy playlist'
    Found 49 playlists
      Fetching tracks from: happy playlist...
        Collected 200 tracks so far...
        Collected 400 tracks so far...
      Fetching tracks from: Happy Playlist...
      Fetching tracks from: pov: ur just happy (playlist)...
      Fetching tracks from: Happy playlist...
        Collected 600 tracks so far...
      Fetching tracks from: ✨HappyPlaylist✨...
      Fetching tracks from: Happy playlist...
        Collected 800 tracks so far...
        Collected 1000 tracks so far...
  Finished collecting 1025 tracks for happy
  💾 Checkpoint saved: 2028 tracks total



  Searching playlists for: 'chill playlist'
    Found 49 playlists
      Fetching tracks from: Chill playlist💤💤💤...
        Collected 200 tracks so far...
      Fetching tracks from: Chill Playlist 💤 Relaxing Vibes 🌌 Slowed...
        Collected 400 tracks so far...
      Fetching tracks from: Chill  Vibes 2026 🌙...
        Collected 600 tracks so far...
      Fetching tracks from: Chill playlist...
        Collected 800 tracks so far...
        Collected 1000 tracks so far...
  Finished collecting 1038 tracks for chill
  💾 Checkpoint saved: 3066 tracks total



  Searching playlists for: 'workout playlist'
    Found 50 playlists
      Fetching tracks from: WORKOUT PLAYLIST 2026...
      Fetching tracks from: GYM SONGS🎀 (for girlies) 2026...
        Collected 200 tracks so far...
      Fetching tracks from: Top 100 Workout Songs In The World...
        Collected 400 tracks so far...
      Fetching tracks from: GYM PHONK 2026 😈 AGGRESSIVE WORKOUT PHON...
        Collected 600 tracks so far...
      Fetching tracks from: MOTIVATION SONGS OF 2026 | BEST SONGS TO...
        Collected 800 tracks so far...
        Collected 1000 tracks so far...
  Finished collecting 1013 tracks for energetic
  💾 Checkpoint saved: 4079 tracks total



  Searching playlists for: 'calm playlist'
    Found 50 playlists
      Fetching tracks from: Calm Songs...
      Fetching tracks from: calm music playlist...
        Collected 200 tracks so far...
        Collected 400 tracks so far...
      Fetching tracks from: My calm playlist ! 🪷...
      Fetching tracks from: calm playlist 🤗🫂....
      Fetching tracks from: ✨️🌙calm playlist🌙✨️...
      Fetching tracks from: Calm playlist🎧...
        Collected 600 tracks so far...
        Collected 800 tracks so far...
      Fetching tracks from: Soft/calm playlist🌊...
        Collected 1000 tracks so far...
  Finished collecting 1070 tracks for calm
  💾 Checkpoint saved: 5149 tracks total



  Searching playlists for: 'party playlist'
    Found 50 playlists
      Fetching tracks from: BEST PARTY PLAYLIST 🎉🎉...
        Collected 200 tracks so far...
      Fetching tracks from: Party Songs 🥳 2010-2026 Hits 🎉...
      Fetching tracks from: Party Playlist...
        Collected 400 tracks so far...
      Fetching tracks from: Party playlist...
        Collected 600 tracks so far...
      Fetching tracks from: DJ Music 2026 🔥Remix Club Party...
        Collected 800 tracks so far...
      Fetching tracks from: AFRO HOUSE MIX 2026 🔥  Party Afro House ...
        Collected 1000 tracks so far...
  Finished collecting 1061 tracks for party
  💾 Checkpoint saved: 6210 tracks total



  Searching playlists for: 'romantic playlist'
    Found 46 playlists
      Fetching tracks from: ROMANTIC SONGS 🌹💘...
      Fetching tracks from: A Romantic Playlist <3...
        Collected 200 tracks so far...
      Fetching tracks from: Top 100 Love Songs 💖 Romantic Playlist |...
        Collected 400 tracks so far...
      Fetching tracks from: POV: U FELL IN LOVE ☁️ ROMANTIC PLAYLIST...
      Fetching tracks from: Romantic playlist...
      Fetching tracks from: Bruno Mars - The Romantic...
      Fetching tracks from: Love and romantic  Tamil songs 🥰...
        Collected 600 tracks so far...
      Fetching tracks from: ROMANTIC INSTRUMENTAL...
        Collected 800 tracks so far...
      Fetching tracks from: Romance books playlist🖤...
        Collected 1000 tracks so far...
  Finished collecting 1063 tracks for romantic
  💾 Checkpoint saved: 7273 tracks total



  Searching playlists for: 'focus playlist'
    Found 48 playlists
      Fetching tracks from: Deep focus study playlist 💻...
      Fetching tracks from: Deep Focus Music...
        Collected 200 tracks so far...
      Fetching tracks from: Work Music 📚 Focus Playlist...
        Collected 400 tracks so far...
      Fetching tracks from: swim focus...
      Fetching tracks from: Deep Focus Playlist 2025 🧠...
      Fetching tracks from: ARIRANG + SWIM FOCUSED (SHORT PLAYLIST)...
      Fetching tracks from: Focus playlist...
        Collected 600 tracks so far...
      Fetching tracks from: Focus Playlist...
        Collected 800 tracks so far...
        Collected 1000 tracks so far...
  Finished collecting 1057 tracks for focus
  💾 Checkpoint saved: 8330 tracks total



Total tracks collected: 8330
